# 🏆 DATATHON 2026 — GenCore v2 | Prophet + LightGBM Residual Hybrid
**Seasonal Naive → Prophet → LGBM Residual → Weighted Ensemble**

In [ ]:
import os, warnings, gc, glob
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

KAGGLE = os.path.exists('/kaggle/input')
if KAGGLE:
    matches = glob.glob('/kaggle/input/**/sales.csv', recursive=True)
    if not matches:
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files[:5]: print(f"  {os.path.join(root, f)}")
        raise FileNotFoundError('sales.csv not found')
    DATA_DIR = os.path.dirname(matches[0])
    OUT_DIR  = '/kaggle/working'
else:
    DATA_DIR = 'data/raw'
    OUT_DIR  = 'output'
print(f"ENV = {'Kaggle' if KAGGLE else 'Local'} | DATA = {DATA_DIR}")

In [ ]:
sales = pd.read_csv(f'{DATA_DIR}/sales.csv')
sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)
sales['doy'] = sales['Date'].dt.dayofyear
sales['dow'] = sales['Date'].dt.dayofweek
sales['year'] = sales['Date'].dt.year

sub_tpl = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
sub_tpl['Date'] = pd.to_datetime(sub_tpl['Date'])
forecast_dates = sub_tpl['Date'].tolist()

print(f"Train: {sales.Date.min().date()} → {sales.Date.max().date()} ({len(sales)} rows)")
print(f"Fcast: {forecast_dates[0].date()} → {forecast_dates[-1].date()} ({len(forecast_dates)} rows)")
print(f"Revenue stats: mean={sales.Revenue.mean():,.0f} std={sales.Revenue.std():,.0f}")

In [ ]:
print("\n═══ TIER 1: Seasonal Naive ═══")

# For each (dayofyear, dayofweek), compute mean from last 3 years
recent = sales[sales['year'] >= 2020].copy()
doy_dow_mean = recent.groupby(['doy','dow'])[['Revenue','COGS']].mean()
doy_mean = recent.groupby('doy')[['Revenue','COGS']].mean()  # fallback

def seasonal_naive_predict(dates):
    preds = []
    for d in dates:
        dt = pd.Timestamp(d)
        key = (dt.dayofyear, dt.dayofweek)
        if key in doy_dow_mean.index:
            preds.append({'Date': d, 'Revenue': doy_dow_mean.at[key,'Revenue'], 'COGS': doy_dow_mean.at[key,'COGS']})
        elif dt.dayofyear in doy_mean.index:
            preds.append({'Date': d, 'Revenue': doy_mean.at[dt.dayofyear,'Revenue'], 'COGS': doy_mean.at[dt.dayofyear,'COGS']})
        else:
            preds.append({'Date': d, 'Revenue': sales.Revenue.mean(), 'COGS': sales.COGS.mean()})
    return pd.DataFrame(preds)

# Validate on 2022
val_mask = sales['year'] == 2022
val_dates = sales.loc[val_mask, 'Date'].tolist()
val_actual = sales.loc[val_mask, 'Revenue'].values

# Build naive using only pre-2022 data for validation
recent_v = sales[(sales['year'] >= 2019) & (sales['year'] <= 2021)]
doy_dow_v = recent_v.groupby(['doy','dow'])[['Revenue','COGS']].mean()
doy_v = recent_v.groupby('doy')[['Revenue','COGS']].mean()

val_naive = []
for d in val_dates:
    dt = pd.Timestamp(d)
    key = (dt.dayofyear, dt.dayofweek)
    if key in doy_dow_v.index: val_naive.append(doy_dow_v.at[key,'Revenue'])
    elif dt.dayofyear in doy_v.index: val_naive.append(doy_v.at[dt.dayofyear,'Revenue'])
    else: val_naive.append(sales.Revenue.mean())
val_naive = np.array(val_naive)

from sklearn.metrics import mean_absolute_error, r2_score
naive_mae = mean_absolute_error(val_actual, val_naive)
naive_r2 = r2_score(val_actual, val_naive)
print(f"Seasonal Naive (val 2022): MAE={naive_mae:,.0f}  R²={naive_r2:.4f}")

In [ ]:
print("\n═══ TIER 2: Prophet ═══")
from prophet import Prophet

# Vietnamese holidays
tet_dates = pd.DataFrame({
    'holiday': 'tet',
    'ds': pd.to_datetime(['2013-02-10','2014-01-31','2015-02-19','2016-02-08',
                          '2017-01-28','2018-02-16','2019-02-05','2020-01-25',
                          '2021-02-12','2022-02-01','2023-01-22','2024-02-10']),
    'lower_window': -3, 'upper_window': 7,
})
sale_events = pd.DataFrame({
    'holiday': 'mega_sale',
    'ds': pd.to_datetime([f'{y}-{m}-{d}' for y in range(2012,2025)
                          for m, d in [('04','30'),('05','01'),('09','02'),('11','11'),('12','12')]
                          if pd.Timestamp(f'{y}-{m}-{d}') <= pd.Timestamp('2024-07-01')]),
    'lower_window': -1, 'upper_window': 1,
})
holidays = pd.concat([tet_dates, sale_events], ignore_index=True)

def fit_prophet(train_df, holidays_df, cap_mult=1.5):
    df = train_df[['Date','Revenue']].rename(columns={'Date':'ds','Revenue':'y'})
    df['cap'] = df['y'].max() * cap_mult
    df['floor'] = 0
    m = Prophet(
        growth='logistic',
        holidays=holidays_df,
        yearly_seasonality=20,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05,
    )
    m.fit(df)
    return m

# Validation: train on pre-2022, predict 2022
train_pre22 = sales[sales['year'] < 2022]
m_val = fit_prophet(train_pre22, holidays)

future_val = pd.DataFrame({'ds': val_dates})
future_val['cap'] = train_pre22['Revenue'].max() * 1.5
future_val['floor'] = 0
fc_val = m_val.predict(future_val)
prophet_val = fc_val['yhat'].clip(lower=0).values

prophet_mae = mean_absolute_error(val_actual, prophet_val)
prophet_r2 = r2_score(val_actual, prophet_val)
print(f"Prophet (val 2022): MAE={prophet_mae:,.0f}  R²={prophet_r2:.4f}")

In [ ]:
print("\n═══ TIER 3: LightGBM Residual ═══")
from lightgbm import LGBMRegressor

# Build SAFE features (no contemporaneous data, no short-term lags)
def build_safe_features(dates_series):
    df = pd.DataFrame({'Date': pd.to_datetime(dates_series)})
    d = df['Date']
    df['month']      = d.dt.month
    df['dayofweek']  = d.dt.dayofweek
    df['dayofyear']  = d.dt.dayofyear
    df['weekofyear'] = d.dt.isocalendar().week.astype(int)
    df['quarter']    = d.dt.quarter
    df['day']        = d.dt.day
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = d.dt.is_month_start.astype(int)
    df['is_month_end']   = d.dt.is_month_end.astype(int)
    df['is_payday']  = df['day'].isin([1,15]).astype(int)
    # Fourier
    for k in [1,2,3]:
        df[f'sin_y{k}'] = np.sin(2*np.pi*k*df['dayofyear']/365.25)
        df[f'cos_y{k}'] = np.cos(2*np.pi*k*df['dayofyear']/365.25)
    for k in [1,2]:
        df[f'sin_w{k}'] = np.sin(2*np.pi*k*df['dayofweek']/7)
        df[f'cos_w{k}'] = np.cos(2*np.pi*k*df['dayofweek']/7)
    # Tet proximity (days until nearest Tet)
    tet_list = tet_dates['ds'].tolist()
    def days_to_tet(dt):
        future_tets = [t for t in tet_list if t >= dt]
        return (future_tets[0] - dt).days if future_tets else 365
    df['days_to_tet'] = df['Date'].apply(days_to_tet)
    df['is_tet_window'] = (df['days_to_tet'] <= 14).astype(int)
    # Historical same-DOY revenue (SAFE: uses only training data)
    doy_rev = recent.groupby('doy')['Revenue'].mean()
    df['hist_doy_revenue'] = df['dayofyear'].map(doy_rev).fillna(sales.Revenue.mean())
    return df.drop(columns=['Date'])

# Train LGBM on prophet residuals (pre-2022)
prophet_train = m_val.predict(pd.DataFrame({
    'ds': train_pre22['Date'],
    'cap': train_pre22['Revenue'].max() * 1.5,
    'floor': 0,
}))
residual_train = train_pre22['Revenue'].values - prophet_train['yhat'].values

X_train_lgb = build_safe_features(train_pre22['Date'])
feat_names = list(X_train_lgb.columns)

lgb_res = LGBMRegressor(
    n_estimators=500, learning_rate=0.03, num_leaves=31, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)
lgb_res.fit(X_train_lgb, residual_train)

# Validate residual model on 2022
X_val_lgb = build_safe_features(pd.Series(val_dates))
residual_pred_val = lgb_res.predict(X_val_lgb)
hybrid_val = (prophet_val + residual_pred_val).clip(min=0)

hybrid_mae = mean_absolute_error(val_actual, hybrid_val)
hybrid_r2 = r2_score(val_actual, hybrid_val)
print(f"Prophet+LGBM (val 2022): MAE={hybrid_mae:,.0f}  R²={hybrid_r2:.4f}")

In [ ]:
print("\n═══ Optimizing Ensemble Weights ═══")
best_w, best_mae = 0, float('inf')
for w_naive in np.arange(0, 0.61, 0.05):
    for w_prophet in np.arange(0, 0.61, 0.05):
        w_hybrid = 1 - w_naive - w_prophet
        if w_hybrid < 0: continue
        blend = w_naive * val_naive + w_prophet * prophet_val + w_hybrid * hybrid_val
        mae = mean_absolute_error(val_actual, blend)
        if mae < best_mae:
            best_mae = mae
            best_w = (w_naive, w_prophet, w_hybrid)

print(f"Best weights: Naive={best_w[0]:.2f} Prophet={best_w[1]:.2f} Hybrid={best_w[2]:.2f}")
print(f"Best blend MAE (val 2022): {best_mae:,.0f}")

In [ ]:
print("\n═══ Retraining on ALL data + Final Forecast ═══")

# 1. Seasonal Naive (uses all data)
naive_forecast = seasonal_naive_predict(forecast_dates)

# 2. Prophet on all data
m_full = fit_prophet(sales, holidays)
future_full = pd.DataFrame({'ds': forecast_dates})
future_full['cap'] = sales['Revenue'].max() * 1.5
future_full['floor'] = 0
fc_full = m_full.predict(future_full)
prophet_forecast_rev = fc_full['yhat'].clip(lower=0).values

# 3. LGBM residual on all data
prophet_all = m_full.predict(pd.DataFrame({
    'ds': sales['Date'], 'cap': sales['Revenue'].max()*1.5, 'floor': 0,
}))
residual_all = sales['Revenue'].values - prophet_all['yhat'].values
X_all_lgb = build_safe_features(sales['Date'])
lgb_full = LGBMRegressor(
    n_estimators=500, learning_rate=0.03, num_leaves=31, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)
lgb_full.fit(X_all_lgb, residual_all)

X_fcast_lgb = build_safe_features(pd.Series(forecast_dates))
residual_fcast = lgb_full.predict(X_fcast_lgb)
hybrid_forecast_rev = (prophet_forecast_rev + residual_fcast).clip(min=0)

# 4. Blend
final_rev = (best_w[0] * naive_forecast['Revenue'].values +
             best_w[1] * prophet_forecast_rev +
             best_w[2] * hybrid_forecast_rev).clip(min=0)

# ─── COGS: same pipeline ───
print("Training COGS pipeline ...")
m_cogs = Prophet(
    growth='logistic', holidays=holidays,
    yearly_seasonality=20, weekly_seasonality=True, daily_seasonality=False,
    seasonality_mode='multiplicative', changepoint_prior_scale=0.05,
)
df_cogs = sales[['Date','COGS']].rename(columns={'Date':'ds','COGS':'y'})
df_cogs['cap'] = df_cogs['y'].max() * 1.5; df_cogs['floor'] = 0
m_cogs.fit(df_cogs)

future_cogs = pd.DataFrame({'ds': forecast_dates})
future_cogs['cap'] = sales['COGS'].max() * 1.5; future_cogs['floor'] = 0
fc_cogs = m_cogs.predict(future_cogs)
prophet_cogs = fc_cogs['yhat'].clip(lower=0).values

residual_cogs_all = sales['COGS'].values - m_cogs.predict(
    pd.DataFrame({'ds': sales['Date'], 'cap': sales['COGS'].max()*1.5, 'floor': 0})
)['yhat'].values
lgb_cogs = LGBMRegressor(
    n_estimators=500, learning_rate=0.03, num_leaves=31, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)
lgb_cogs.fit(X_all_lgb, residual_cogs_all)
hybrid_cogs = (prophet_cogs + lgb_cogs.predict(X_fcast_lgb)).clip(min=0)
final_cogs = (best_w[0] * naive_forecast['COGS'].values +
              best_w[1] * prophet_cogs +
              best_w[2] * hybrid_cogs).clip(min=0)

In [ ]:
submission = sub_tpl[['Date']].copy()
submission['Revenue'] = np.round(final_rev, 2)
submission['COGS']    = np.round(final_cogs, 2)
submission['Date']    = submission['Date'].dt.strftime('%Y-%m-%d')

os.makedirs(OUT_DIR, exist_ok=True)
out_path = f'{OUT_DIR}/submission.csv'
submission.to_csv(out_path, index=False, float_format='%.2f')

print(f"\n✅ Submission saved: {out_path}")
print(f"   Rows: {len(submission)}")
print(f"   Revenue: mean={submission.Revenue.mean():,.0f}  std={submission.Revenue.std():,.0f}")
print(f"   COGS:    mean={submission.COGS.mean():,.0f}  std={submission.COGS.std():,.0f}")
print(submission.head(10))

In [ ]:
assert len(submission) == len(sub_tpl), f"Row mismatch!"
assert submission.Revenue.isna().sum() == 0, "NaN Revenue!"
assert submission.COGS.isna().sum() == 0, "NaN COGS!"
assert (submission.Revenue >= 0).all(), "Negative Revenue!"
assert (submission.COGS >= 0).all(), "Negative COGS!"

print("\n✅ All checks passed!")
print(f"\n{'='*50}")
print(f"VALIDATION SUMMARY (2022 holdout):")
print(f"  Seasonal Naive : MAE={naive_mae:>12,.0f}  R²={naive_r2:.4f}")
print(f"  Prophet         : MAE={prophet_mae:>12,.0f}  R²={prophet_r2:.4f}")
print(f"  Prophet+LGBM    : MAE={hybrid_mae:>12,.0f}  R²={hybrid_r2:.4f}")
print(f"  Best Blend      : MAE={best_mae:>12,.0f}")
print(f"  Weights: Naive={best_w[0]:.2f} Prophet={best_w[1]:.2f} Hybrid={best_w[2]:.2f}")
print(f"{'='*50}")